# 02 – Feature Engineering

Pipeline de construcción de variables para los modelos predictivos.

## Objetivos

1. Construir features ricos: calendario, lags, rolling, exógenas.
2. Justificar la codificación cíclica de variables horarias.
3. Visualizar la importancia inicial de las features (correlación con target).
4. Realizar la partición temporal train / val / test.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

from tfm_energia.features.feature_builder import (
    FeatureBuilder,
    FeatureConfig,
    cargar_features_sede,
)

SEDE = "madrid"

## 1. Construir features para Madrid

In [ ]:
df_feat = cargar_features_sede(SEDE)
print(f"Shape final: {df_feat.shape}")
print(f"Periodo: {df_feat.index.min()} → {df_feat.index.max()}")
df_feat.head(3)

In [ ]:
# Listar las features creadas
builder = FeatureBuilder()
features = builder.get_feature_cols(df_feat)
print(f"Total features creadas: {len(features)}")
print("\nGrupos de features:")
print(f"  Calendario: {[c for c in features if not any(k in c for k in ['lag', 'roll', '_c', '_kwh', 'sede'])][:15]}")
print(f"  Cíclicas: {[c for c in features if c.endswith('_sin') or c.endswith('_cos')]}")
print(f"  Lags consumo: {[c for c in features if 'consumo_total' in c and 'lag' in c]}")
print(f"  Rolling consumo: {[c for c in features if 'consumo_total' in c and 'roll' in c]}")

## 2. Visualización de codificación cíclica

In [ ]:
# Demostración visual de por qué hora_sin/cos > hora directa
horas = np.arange(24)
sin = np.sin(2 * np.pi * horas / 24)
cos = np.cos(2 * np.pi * horas / 24)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(horas, horas, marker='o', label='hora directa')
ax1.set_title("Sin codificación: 23h y 0h muy lejos")
ax1.set_xlabel("Hora real")
ax1.set_ylabel("Valor numérico")
ax1.grid(True)

ax2.plot(sin, cos, marker='o')
for h in [0, 6, 12, 18, 23]:
    ax2.annotate(f'{h}h', (sin[h], cos[h]), xytext=(5,5), textcoords='offset points')
ax2.set_title("Con codificación cíclica: 23h y 0h adyacentes")
ax2.set_xlabel("hora_sin")
ax2.set_ylabel("hora_cos")
ax2.axis('equal')
ax2.grid(True)
plt.tight_layout()
plt.show()

## 3. Correlación de features con el target

In [ ]:
target = "consumo_total_kwh"
feat_num = df_feat[features].select_dtypes(include=[np.number])
corrs = feat_num.corrwith(df_feat[target]).sort_values(ascending=False)

# Top 20 positivas + top 10 negativas
top = pd.concat([corrs.head(20), corrs.tail(10)])

fig, ax = plt.subplots(figsize=(10, 9))
colors = ['steelblue' if x > 0 else 'crimson' for x in top.values]
ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1])
ax.set_xlabel("Correlación con consumo_total_kwh")
ax.set_title("Top features más correlacionadas con el consumo")
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 4. Particionado temporal train / val / test

In [ ]:
train, val, test = builder.split_temporal(df_feat, train_frac=0.7, val_frac=0.15)

# Visualizar split
fig, ax = plt.subplots(figsize=(15, 4))
for name, sub, color in [
    ("Train", train, "steelblue"),
    ("Validation", val, "orange"),
    ("Test", test, "crimson"),
]:
    sub_daily = sub[target].resample('D').sum()
    ax.plot(sub_daily.index, sub_daily.values, label=f"{name} ({len(sub):,} h)", color=color, linewidth=0.8)
ax.set_title("Partición temporal train/val/test (consumo diario agregado)")
ax.set_ylabel("kWh / día")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Sanity check: no hay NaN ni leakage

In [ ]:
# Verificar ausencia de NaN en features
nan_count = df_feat[features].isna().sum().sum()
print(f"NaN en features: {nan_count}")

# Verificar que rolling no tiene lookahead: en posición i, rolling solo usa info hasta i-1
muestra = df_feat[['consumo_total_kwh', 'consumo_total_kwh_roll_mean_3h']].head(10)
muestra['esperado'] = df_feat['consumo_total_kwh'].rolling(3).mean().shift(1).head(10).values
muestra

## 6. Guardar splits para los siguientes notebooks

In [ ]:
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)

train.to_parquet(PROCESSED / f'train_{SEDE}.parquet')
val.to_parquet(PROCESSED / f'val_{SEDE}.parquet')
test.to_parquet(PROCESSED / f'test_{SEDE}.parquet')

print(f"Splits guardados en {PROCESSED}")
for fname in [f'train_{SEDE}.parquet', f'val_{SEDE}.parquet', f'test_{SEDE}.parquet']:
    size_mb = (PROCESSED / fname).stat().st_size / 1024 / 1024
    print(f"  {fname}: {size_mb:.2f} MB")

## 7. Próximos pasos

Con los splits guardados, en el notebook **03** entrenaremos:

1. **Baseline naïve estacional** (último valor de la misma hora hace 24h o 168h).
2. **SARIMAX** con variables exógenas (T exterior, ocupación).
3. **Prophet** con regresores.
4. **LSTM** multivariante con secuencias.

Y compararemos los cuatro en métricas: MAE, RMSE, MAPE, sMAPE.